<a href="https://colab.research.google.com/github/tavishgrade/nlp-language-model-portfolio/blob/main/Talktorial_1_Lexical_TavishGrade.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Talktorial 1 — Lexical: Build a Bigram Language Model & PMI Collocation Finder

**Graduate NLP · Week 2 · Exposure #2** (lecture → **talktorial** → peer discussion)

In the lecture you *saw* n-gram models and lexical association. Here you **build** them
on a real text — *Alice's Adventures in Wonderland* — with an AI assistant as your tutor,
not your ghostwriter.

> ⚠️ **Colab users — do this first:** **File → Save a copy in Drive.**
> The runtime resets when idle, and unsaved work (and downloaded data) is lost.

**By the end you will be able to:**
1. **Tokenize** raw text into words and sentences
2. **Build** a bigram language model from counts
3. **Evaluate** it with **perplexity** on held-out text
4. **Find** collocations with **PMI** and log-likelihood
5. **Verify** an AI assistant's claims against authoritative sources

**How you'll be graded:** answer the prediction questions below, run every cell (outputs must be
saved), complete the **AI Verification Log**, answer the **three reflection questions** at the end,
then submit this `.ipynb` to Canvas by **Sun 11:59 PM**.

## Before you code: make some predictions

Answering these *first* turns the code below into a test of your intuition.
Type your answers in this cell (double-click to edit):

- **Q1.** Which single word do you predict is **most frequent** in Alice, and roughly what
  fraction of all words is it?
  > *Your answer:* "the" - nearly 17%. Randomly educated guess - intrigued to see the answer.
- **Q2.** Why might a bigram model assign probability **zero** to a perfectly grammatical
  sentence?
  > *Your answer:* If the sentence contains unique word pairings not found in a corpus, then that result of zero would affect the entire probability of the sentence through multiplication, making it zero
- **Q3.** Name **two words** you bet co-occur in Alice far more than chance. We'll test it in Part 4.
  > *Your answer:* cheshire cat - I would assume that's not a historically common pairing

## Part 0 — Setup

This cell is provided. It re-downloads the corpus (safe to re-run after a runtime reset).

In [ ]:
import nltk
for pkg in ["gutenberg", "punkt", "punkt_tab", "stopwords"]:
    nltk.download(pkg, quiet=True)

from nltk.corpus import gutenberg
from nltk.tokenize import sent_tokenize, word_tokenize
from collections import Counter
from nltk.collocations import BigramCollocationFinder, BigramAssocMeasures
import math, json

print("Setup complete.")

Setup complete.


## Part 1 — Load & tokenize *Alice*

Provided: load the raw text.

In [ ]:
raw = gutenberg.raw("carroll-alice.txt")
print(raw[:500])
print("\n... Total characters:", len(raw))

[Alice's Adventures in Wonderland by Lewis Carroll 1865]

CHAPTER I. Down the Rabbit-Hole

Alice was beginning to get very tired of sitting by her sister on the
bank, and of having nothing to do: once or twice she had peeped into the
book her sister was reading, but it had no pictures or conversations in
it, 'and what is the use of a book,' thought Alice 'without pictures or
conversation?'

So she was considering in her own mind (as well as she could, for the
hot day made her feel very sleepy an

... Total characters: 144395


In [ ]:
# TODO: Turn `raw` into `sentences` (a list of token-lists) and `tokens` (one flat list).
# Hints:
#   - sent_tokenize(raw)      -> list of sentence strings
#   - word_tokenize(sentence) -> list of word-tokens
#   - lowercase every token
# Do NOT just split on spaces -- remember from lecture why that fails on "don't", "U.S.A.", etc.

# split raw book data into a list of separate sentence strings
sentence_strings = sent_tokenize(raw)

# split list of sentence strings into list of tokens from each sentence, lowercasing each token in the process
sentences = []
for s in sentence_strings:
    s_tokens = word_tokenize(s)
    s_tokens = [t.lower() for t in s_tokens]
    sentences.append(s_tokens)

# combine each sentence token lists into one full list of word tokens, maintaining lower case to eliminate case sensitive duplicates
tokens = [t for sent in sentences for t in sent]

print("Sentences:", len(sentences))
print("Tokens:   ", len(tokens))

# verify lowercase and tokenization method was successful
print(sentences[0])
print(tokens[:30])

Sentences: 1625
Tokens:    33535
['[', 'alice', "'s", 'adventures', 'in', 'wonderland', 'by', 'lewis', 'carroll', '1865', ']', 'chapter', 'i', '.']
['[', 'alice', "'s", 'adventures', 'in', 'wonderland', 'by', 'lewis', 'carroll', '1865', ']', 'chapter', 'i', '.', 'down', 'the', 'rabbit-hole', 'alice', 'was', 'beginning', 'to', 'get', 'very', 'tired', 'of', 'sitting', 'by', 'her', 'sister', 'on']


In [ ]:
# TODO: How many tokens? How many unique types? Compute the type/token ratio.
# Then print the 10 most common tokens -- does your Q1 prediction hold?
# Hints: len(...), set(...), Counter(tokens).most_common(10)

# calculate the amount of tokens from Alice in Wonderland
num_tokens = len(tokens)
# calculate the number of unique tokens from Alice in Wonderland
num_types  = len(set(tokens))
# calculate the type/token ratio to indicate lexical diversity
type_token_ratio = num_types / num_tokens

# print out results
print(num_tokens)
print(num_types)
print(type_token_ratio)

# calculate and print the most common tokens
freqs = Counter(tokens)
print(freqs.most_common(10))

# calculate how often 'the' appears in the text (my guess for highest)
count_the = freqs["the"]
percentage_the = count_the / num_tokens
print(percentage_the)

# within the top ten tokens, I see the second highest (the) was my prediction
# but the top token was a punctuation, and I hadn't even considered punctuation
# I see that my percentage idea of how often 'the' would be (17%) was much higher than the actual (5%)


33535
2808
0.08373341285224392
[(',', 2418), ('the', 1616), ("'", 1311), ('.', 975), ('and', 810), ('to', 720), ('a', 631), ('she', 545), ('i', 543), ('it', 540)]
0.04818845981810049


In [ ]:
# TODO: Wrap each sentence in `sentences` with "<s>" at the front and "</s>" at the end.
# (Lecture: why do the first and last words need these?)  Store as `padded`.

# add padding to the beginning and ending of each sentence
padded = []
for sent in sentences:
  padded.append(["<s>"] + sent + ["</s>"])

# print the tenth sentence with the added padding to inspect result
print(padded[10])

# adding a padding to the beginning and ending of our sentences informs the model of sentence structure

['<s>', 'first', ',', 'she', 'tried', 'to', 'look', 'down', 'and', 'make', 'out', 'what', 'she', 'was', 'coming', 'to', ',', 'but', 'it', 'was', 'too', 'dark', 'to', 'see', 'anything', ';', 'then', 'she', 'looked', 'at', 'the', 'sides', 'of', 'the', 'well', ',', 'and', 'noticed', 'that', 'they', 'were', 'filled', 'with', 'cupboards', 'and', 'book-shelves', ';', 'here', 'and', 'there', 'she', 'saw', 'maps', 'and', 'pictures', 'hung', 'upon', 'pegs', '.', '</s>']


## Part 2 — Build the bigram language model

In [ ]:
# TODO: Build two Counters:
#   unigrams[w]        = how often word w appears (over the PADDED tokens)
#   bigrams[(w1, w2)]  = how often the pair (w1, w2) appears adjacently
# Hint for bigrams: iterate each padded sentence and use zip(s, s[1:]).

unigrams = Counter()
bigrams = Counter()

# iterate through the padded list of sentence tokens
for sent in padded:
  # first count unigram totals and add into empty unigrams Counter
  for w in sent:
    unigrams[w] += 1

  # then count the bigram totals and add into empty bigrams Counter
  for w1, w2 in zip(sent, sent[1:]):
    bigrams[(w1, w2)] += 1

# test through some examples of text to verify results
# this test should equal the len of sentences
print(unigrams["<s>"])
print(len(sentences))

# this test should equal the same as box 11 'the' frequency result
print(unigrams["the"])
print(count_the)

# test how often 'Alice' is the beginning of a sentence
print(bigrams["<s>","alice"])

# test how often 'she' 'said' appear next to each other
print(bigrams["she","said"])

# test how often 'rabbit' is the end of a sentence
print(bigrams["rabbit","</s>"])


1625
1625
1616
1616
80
34
0


In [ ]:
# TODO: Write bigram_prob(prev, w) = C(prev, w) / C(prev)   (the lecture formula).
# Guard against dividing by zero.

def bigram_prob(prev, w):
  # first check if prev has a count of zero to prevent mathematical errors dividing by zero
  # return probability correctly as zero if prev has count of zero
  if unigrams[prev] == 0:
    return 0.0

  # after checking for potential mathematical errors, continue finding probability
  # return probability of w being found after prev in corpus
  return bigrams[prev, w] / unigrams[prev]

print("P(rabbit | white) =", bigram_prob("white", "rabbit"))

# and this is the moment I realized red queen is not in the original book
print("P(queen | red) =", bigram_prob("red", "queen"))

print("P(cat | cheshire) =", bigram_prob("cheshire", "cat"))

P(rabbit | white) = 0.7333333333333333
P(queen | red) = 0.0
P(cat | cheshire) = 0.8333333333333334


In [ ]:
# TODO: Compute P("the white rabbit") by multiplying bigram probabilities along the
# (padded) sentence. Then compute it again by SUMMING log-probabilities.
# Why is the log version necessary for long sentences?  (Lecture: underflow.)

sent = ["<s>", "the", "white", "rabbit", "</s>"]

# find the probability by multiplying bigram probabilites along the sentence
raw_prob = 1.0
for w1, w2 in zip(sent, sent[1:]):
  raw_prob *= bigram_prob(w1, w2)

# print out probability result to verify
print("Raw probability = ", raw_prob)

# now calculate the sum of log-probabilities to prevent underflow
# while preventing log zero errors by providing the calculation of log zero to be negative infinity
log_prob = 0.0
for w1, w2 in zip(sent, sent[1:]):
  p = bigram_prob(w1, w2)
  # print out each probability for testing purposes
  print(p)
  log_prob += math.log(p) if p > 0 else float('-inf')

# print out log probability result to verify and compare
print("Log probability = ", log_prob)

# the log version is necessary for long sentences because of the higher chance of finding new pairings
# with multiplication of probabilities, virtually all results will lead to numbers too small
# for Python for correctly display without underflowing down to zero


Raw probability =  0.0
0.06523076923076923
0.013613861386138614
0.7333333333333333
0.0
Log probability =  -inf


### ✅ Verification checkpoint #1
Ask your AI assistant: *"How do I compute a bigram probability from counts?"*
**Verify** its answer against the **Lexical lecture** and **SLP3 §N-Grams**, then record it in
the Verification Log (Part 5).

## Part 3 — Smoothing & perplexity

In [ ]:
# TODO: Write bigram_prob_laplace(prev, w) = (C(prev,w) + 1) / (C(prev) + V).
# What is V here?  (Lecture: why do we add V to the denominator?)

# compute V as the count of unique tokens by calculating len of unigrams
V = len(unigrams)
# compute laplace smoothing probability
def bigram_prob_laplace(prev, w):
  return (bigrams[prev, w] + 1) / (unigrams[prev] + V)

# print test probability
print(bigram_prob_laplace("white", "rabbit"))

# test presents a probability much lower than previous unsmooth testing


0.008098591549295774


Provided: an honest held-out split — train on the first 90% of sentences, test on the last 10%.

In [ ]:
split = int(0.9 * len(padded))
train_sents, test_sents = padded[:split], padded[split:]
print("Train sentences:", len(train_sents), "| Test sentences:", len(test_sents))
# (A chapter-based split is a nice extension -- see the wrap-up.)

Train sentences: 1462 | Test sentences: 163


In [ ]:
# TODO: Measure perplexity of the SMOOTHED model on the held-out test set:
#        PP = exp( -(1/N) * sum of log P(w_i | w_{i-1}) )  over all test bigrams.
# Then try the UNSMOOTHED model -- what value do you get, and why?
# IMPORTANT: build the model's counts from `train_sents` only (honest evaluation).

# create data variables of stricly the training data, a repeat of the earlier exercise
train_unigrams = Counter()
train_bigrams = Counter()

# fill training data libraries with token data
for sent in train_sents:
  for w in sent:
    train_unigrams[w] += 1
  for w1, w2 in zip(sent, sent[1:]):
    train_bigrams[w1, w2] += 1

# create laplace probability function of training data
# start with finding count of unique tokens in training data
V = len(train_unigrams)

# continue with creating laplace smoothing probability function
def bigram_prob_laplace(prev, w):
  return (train_bigrams[prev, w] + 1) / (train_unigrams[prev] + V)

# now calculate perplexity of the model on the test data
log_sum = 0.0
N = 0

# fill the variables log_sum and N through each bigram set of the test data using the model
for sent in test_sents:
  for w1, w2 in zip(sent, sent[1:]):
    p = bigram_prob_laplace(w1, w2)
    log_sum += math.log(p)
    N += 1

# now determine the perplexity
smoothed_perplexity = math.exp(-log_sum / N)
# print out result
print("Smoothed perplexity = ", smoothed_perplexity)

# now test perplexity on the unsmoothed model
def bigram_prob_unsmoothed(prev, w):
  if train_unigrams[prev] == 0:
    return 0.0
  return train_bigrams[prev, w] / train_unigrams[prev]

# now calculate perplexity of the model on the test data
log_sum = 0.0
N = 0

# fill the variables log_sum and N through each bigram set of the test data using the unsmoothed model
for sent in test_sents:
  for w1, w2 in zip(sent, sent[1:]):
    p = bigram_prob_unsmoothed(w1, w2)
    if p == 0:
      log_sum += float('-inf')
    else:
      log_sum += math.log(p)
    N += 1

# now determine the perplexity
unsmoothed_perplexity = math.exp(-log_sum / N)
# print out result
print("Unsmoothed perplexity = ", unsmoothed_perplexity)

# when comparing the smoothed and unsmoothed it's clear the difference
# unsmoothed will virtually always result in positive infinity, and while the laplace
# smoothing resulted in a higher number, it's something substantially lower than unsmoothed

# unsmoothed results in the infinity because there is bound to be a bigram pairing not seen
# in the training data, resulting in a zero. A log of zero equals -infinity.

Smoothed perplexity =  515.1400748451176
Unsmoothed perplexity =  inf


### ✅ Verification checkpoint #2
Ask: *"What does perplexity measure, and is lower better?"*
Verify against **SLP3 §Perplexity** and the **Lexical lecture**. Record it in Part 5.

## Part 4 — Find collocations with PMI
This is the Python version of the association measures from the lecture (PMI, log-likelihood).

In [ ]:
# TODO: Use BigramCollocationFinder + BigramAssocMeasures to rank word pairs by PMI.
# Steps:
words = [w for w in tokens if w.isalpha()]
measures = BigramAssocMeasures()
finder = BigramCollocationFinder.from_words(words)
finder.nbest(measures.pmi, 20)


# Look at the top pairs -- do they look meaningful, or like noise?
# the top pairs are a mixture of very unique pairings special to the book and some proper names
# they definitely feel like noise, no pairings that I correlate to the book from memory


[('abide', 'figures'),
 ('ambition', 'distraction'),
 ('arithmetic', 'ambition'),
 ('barking', 'hoarsely'),
 ('bathing', 'machines'),
 ('beauti', 'ful'),
 ('brother', 'latin'),
 ('buttered', 'toast'),
 ('canvas', 'bag'),
 ('circle', 'exact'),
 ('crocodile', 'improve'),
 ('custard', 'roast'),
 ('deepest', 'contempt'),
 ('draggled', 'feathers'),
 ('edgar', 'atheling'),
 ('elsie', 'lacie'),
 ('energetic', 'remedies'),
 ('esq', 'hearthrug'),
 ('exact', 'shape'),
 ('fair', 'warning')]

In [ ]:
# TODO: Apply a minimum-frequency filter (finder.apply_freq_filter(3)) and re-rank by PMI.
# Save the result as `top_pmi`. Did your Q3 pair (e.g. "white rabbit") surface now?

# apply a minimum-frequency filter
finder.apply_freq_filter(3)

# rerank by PMI
top_pmi = finder.nbest(measures.pmi, 20)

# print out results to verify
print(top_pmi)

# I do not see white rabbit yet
# but I changed the frequency threshold to 10 to test and immediately saw white rabbbit

[('beau', 'ootiful'), ('yer', 'honour'), ('mary', 'ann'), ('young', 'lady'), ('ootiful', 'soo'), ('lobster', 'quadrille'), ('soo', 'oop'), ('play', 'croquet'), ('e', 'evening'), ('golden', 'key'), ('fast', 'asleep'), ('evening', 'beautiful'), ('kid', 'gloves'), ('father', 'william'), ('e', 'e'), ('four', 'times'), ('twinkle', 'twinkle'), ('dead', 'silence'), ('few', 'minutes'), ('broken', 'glass')]


In [ ]:
# TODO: Re-rank the SAME filtered finder by measures.likelihood_ratio; save as `top_ll`.
# How do the top pairs differ from PMI's?  Which list looks more sensible to you?
top_ll = finder.nbest(measures.likelihood_ratio, 20)

print(top_ll)

# it seems this likelihood ratio list lists pairings significantly more common in the text
# it reflects accurately the prevelance of the pairings against independence
# whereas the PMI list shows pairings unique to this text and measures common words in a pair low
# removing common interchangeable pairings such as "said" "the" shown in the ll list

[('mock', 'turtle'), ('said', 'the'), ('said', 'alice'), ('march', 'hare'), ('went', 'on'), ('the', 'queen'), ('the', 'king'), ('in', 'a'), ('a', 'little'), ('the', 'gryphon'), ('the', 'mock'), ('the', 'hatter'), ('white', 'rabbit'), ('you', 'know'), ('she', 'had'), ('it', 'was'), ('to', 'herself'), ('the', 'duchess'), ('as', 'she'), ('did', 'not')]


In [ ]:
# TODO: The log-likelihood list above is full of stopword pairs ("said the", "in a").
# Build an English stoplist and drop every bigram that contains a stopword, then re-rank.
# Hints:

from nltk.corpus import stopwords
stop = set(stopwords.words("english"))
# drops pairs containing a stopword
finder.apply_word_filter(lambda w: w in stop)
# rerank the likelihood ratios
top_ll_clean = finder.nbest(measures.likelihood_ratio, 20)

# print top list to verify results
print(top_ll_clean)

# Compare with the previous list -- which is more useful? (Lecture: "tokenization choices revisited".)
# a clean stopword free list bring out more unique findings from the book
# from initial observations it feels a good mix between PMI and LL initial findings
# the list highlights real content of the book rather than grammar commonalities


[('mock', 'turtle'), ('said', 'alice'), ('march', 'hare'), ('white', 'rabbit'), ('thought', 'alice'), ('soo', 'oop'), ('golden', 'key'), ('good', 'deal'), ('evening', 'beautiful'), ('kid', 'gloves'), ('play', 'croquet'), ('beau', 'ootiful'), ('yer', 'honour'), ('three', 'gardeners'), ('beautiful', 'soup'), ('ootiful', 'soo'), ('oh', 'dear'), ('lobster', 'quadrille'), ('white', 'kid'), ('great', 'hurry')]


### ✅ Verification checkpoint #3
Ask: *"What does `BigramAssocMeasures.pmi` actually compute?"*
Verify against the **NLTK documentation** and **SLP3 Ch.6 §PMI**. Record it in Part 5.

## Part 5 — AI Verification Log

For each checkpoint, name the **source** you checked and your **verdict**. Add at least one
claim of your own. (This is what proves you *engaged* with the AI rather than pasting its output.)


> **Filling this in:** double-click this cell to edit it, type between the `|` characters, then press
> **Shift+Enter** to render it again. Keep each note to a sentence, and avoid typing `|` inside a
> note — it splits the row into an extra column. (Works the same in Colab, Jupyter, and VS Code.)

| # | Claim the AI made | Source checked | Verdict (✅ / ⚠️ / ❌) | Your note |
|---|---|---|---|---|
| 1 | "Bigram prob = C(prev,w) / C(prev)" | Lecture / SLP3 §N-Grams | ✅ | The AI claim provided the final equation written here in this claim, but the SLP book chapter 3 first introduced the expanded version where the denominator is the sum of all the bigrams starting with prev word but is then simplified to what you wrote here |
| 2 | "Lower perplexity = better model" | Lecture / SLP3 §Perplexity | ✅ | All three sources verify that lower is better, making the case that as the probabilties get higher, the denominator of the perplexity equation gets larger, resulting in a smaller perplexity |
| 3 | "PMI = log₂( P(x,y) / (P(x)·P(y)) )" | Lecture / NLTK doc / SLP3 | ✅ | In the lecture and book, the equation is displayed the same as the AI. Further definition from the Lecture and the AI describe PMI as how much more often two words occur together than by chance |
| 4 | "PMI is biased against stopwords" | Lecture / NLTK doc | ✅ | Both sources relay similar information that rare words create a lower denominator, skyrocketing the PMI value higher, thus the need for NLTK stopword library |

**Verdict key:** ✅ correct · ⚠️ partly right / needs caveat · ❌ wrong.

## Part 6 — Save your artifacts (for the final project)

In [ ]:
# TODO: Build a dict `artifact` in the shared envelope:
#   {"week": 2, "task": "lexical-ngrams", "source": ..., "outputs": {...}}
# Put num_tokens, vocab_size, heldout_perplexity, and your top PMI /
# log-likelihood / stopword-filtered collocation lists inside "outputs".
# Save it as talktorial1_artifact.json. This chains into the final project.
artifact = {
    "week": 2,
    "task": "lexical-ngrams",
    "source": "carroll-alice.txt",
    "outputs": {
        "num_tokens": num_tokens,
        "vocab_size": num_types,
        "heldout_perplexity": smoothed_perplexity,
        "top_pmi": top_pmi,
        "top_ll": top_ll,
        "top_ll_clean": top_ll_clean
    }
}

with open("talktorial1_artifact.json", "w") as f:
  json.dump(artifact, f, indent=2)


## Before you submit — three questions, honestly answered

These are not graded for being impressive. They are graded for being real, and they are
how I find out what is not landing while there is still time to do something about it.
*(Double-click this cell to type your answers, then Shift+Enter.)*

**Which AI did you use this week?** Name the assistant, and the model if you know it — e.g.
"ChatGPT", "Claude", "Gemini", "Copilot", a local model, or "none this week", which is a
perfectly good answer. This is not graded; I ask because what these tools get wrong differs
between them, and it shapes what I cover.

> *Your answer:* Co-Pilot

**1. What surprised you?**
*(Scroll back to your predictions at the top — where was your intuition wrong, and why?)*

> *Your answer:* When we first worked with Leplace smoothing, I was still trying to understand the concept (although looking back to the lecture slide referencing Robin Hood was helpful), so the substancially different probability number for rabbit | white was a bit shocking. It made me check my answer twice before moving on.
I realize now that you mention the top predictions - I'll leave the prior answer just for extra. I was surprised at the highest counted token being a punctuation mark. I didn't even register that as an option in the initial predictions, moving straight into common filler english words.

**2. What got you stuck this week, and what did you do about it?**
*("I am still stuck on X" is a complete and fully-credited answer.)*

> *Your answer:* Actually in the very first task I got stuck trying to figure out how to correctly token out a text by hand. It took some critical thinking before I copy and pasted the question into google and the documentation for the imported functions was the first result. I didn't immediately register those as imported functions and assumed I had to create my own tokenize function on the very first task! Reading through the NLTK docs got me feeling more prepared to start the tasks.

**3. What would you ask if we were in a room together?**

> *Your answer:* Considering the variation of choices between PMI and LogLikelihood ratio or between tokenization functions other than the tokenize we used - I understand your lessons are on the subtleties of these variations - do we then make education choices between varities specifically on our needs of the project? Such as using PMI could show how inventive certain pairings were from Carrol's writings, but LL ratio could show common pairings from that time period and wouldn't mind having stop words appear. Aka - there's no one right answer but it's based on our initial question we're trying to solve for?


## Wrap-up

You built a bigram language model, evaluated it with perplexity, and found collocations with
PMI — the same word-prediction idea that scales up to GPT and Claude (neural nets + learned
vectors instead of count tables). Next stop: **Week 6**, where PMI becomes **word2vec**.

**Optional extensions:** split train/test by *chapter* instead of by sentence; try trigrams;
add your own domain-specific words on top of the NLTK stoplist.

**Submit:** Run all cells (so outputs are saved), then **File → Download .ipynb** and upload to
Canvas by **Sun 11:59 PM**, with the Verification Log completed.